In [1]:
####################################################
####################################################
## test the motion mapper script converted from Matlab 
## original code from https://github.com/PrincetonUniversity/MouseMotionMapper/tree/main
## converted by AI 
####################################################
####################################################

In [6]:
from __future__ import annotations

import os
import sys
import numpy as np

import matplotlib
import skimage

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from helpers import *

#import mousemotionmapper as mmm

In [ ]:
#sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))


In [7]:
def make_synthetic_recording(n_frames, n_bodyparts=17, fs=80.0, rng=None):
    """Create one synthetic recording of 2-D body-part trajectories.

    The mouse is modelled as a set of body parts whose relative geometry
    oscillates according to one of several motifs (different frequencies), so
    the distance/wavelet features carry real structure for t-SNE to separate.
    """
    rng = np.random.default_rng() if rng is None else rng
    t = np.arange(n_frames) / fs

    # Base resting layout of the body parts (roughly along the body axis).
    base = np.zeros((n_bodyparts, 2))
    base[:, 0] = np.linspace(-5, 5, n_bodyparts)          # x along the body
    base[:, 1] = 0.5 * rng.standard_normal(n_bodyparts)   # small y offsets

    coords = np.zeros((n_bodyparts, 2, n_frames))

    # Three motifs with distinct oscillation frequencies (Hz).
    motif_freqs = [1.5, 5.0, 12.0]
    seg_len = n_frames // 6
    labels = np.zeros(n_frames, dtype=int)

    pos = 0
    while pos < n_frames:
        motif = rng.integers(0, len(motif_freqs))
        f = motif_freqs[motif]
        end = min(pos + seg_len, n_frames)
        labels[pos:end] = motif
        local_t = t[pos:end]
        # Each body part wiggles with a part-specific phase & amplitude.
        for b in range(n_bodyparts):
            amp = 0.6 + 0.4 * (b / n_bodyparts)
            phase = 2 * np.pi * b / n_bodyparts
            coords[b, 0, pos:end] = base[b, 0] + amp * np.sin(2 * np.pi * f * local_t + phase)
            coords[b, 1, pos:end] = base[b, 1] + amp * np.cos(2 * np.pi * f * local_t + phase)
        pos = end

    # A little tracking noise.
    coords += 0.05 * rng.standard_normal(coords.shape)
    return coords, labels


In [8]:
rng = np.random.default_rng(0)

# ------------------------------------------------------------------ params
params = mmm.RunParameters()    

# Keep the synthetic demo fast but representative.
params.pcaModes = 6
params.numProjections = 6
params.numPeriods = 15
params.trainingSetSize = 60          # templates kept per file
params.training_numPoints = 1500     # sub-sample size per file
params.samplingFreq = 80.0
params.minF = 0.25
params.maxF = 20.0
params.perplexity = 30.0
params.tsne_seed = 0

NameError: name 'mmm' is not defined

In [9]:
# --------------------------------------------------------------- synthesise
n_files = 5
coord_list = []
label_list = []
for _ in range(n_files):
    coords, labels = make_synthetic_recording(4000, n_bodyparts=17, fs=80.0, rng=rng)
    coord_list.append(coords)
    label_list.append(labels)
print(f"Created {n_files} synthetic recordings of shape {coord_list[0].shape} "
          f"(bodyparts, dims, frames)")

# Use the first 11 body parts for distances (demo subset).
bodypart_indices = np.arange(11)


Created 5 synthetic recordings of shape (17, 2, 4000) (bodyparts, dims, frames)


In [10]:
bodypart_indices

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])

In [14]:
#print(coord_list)
print(label_list)

[array([0, 0, 0, ..., 1, 1, 1], shape=(4000,)), array([0, 0, 0, ..., 1, 1, 1], shape=(4000,)), array([1, 1, 1, ..., 1, 1, 1], shape=(4000,)), array([0, 0, 0, ..., 0, 0, 0], shape=(4000,)), array([2, 2, 2, ..., 0, 0, 0], shape=(4000,))]


In [ ]:
# ----------------------------------------------------------- run pipeline for mmm package
"""End-to-end: coordinates -> training-set t-SNE embedding.

    Returns
    -------
    dict with keys:
        distances   : list of per-file distance matrices
        pca         : fitted OnlinePCA
        data_all    : training-set feature matrix (dataAll)
        embedding   : (n_training, 2) final t-SNE embedding (ydata)
    """

result = mmm.run_full_pipeline(
    coord_list,
    bodypart_indices=bodypart_indices,
    parameters=params,
    rng=rng,
    verbose=True,
)
distances = result["distances"]
pca = result["pca"]
data_all = result["data_all"]
embedding = result["embedding"]

In [ ]:
# ----------------------------------------------------------- 
# dissect each step of 
# ----------------------------------------------------------- 

In [ ]:
# ----------------------------------------------------------- run pipeline for mmm package
parameters = set_run_parameters(parameters)
rng = np.random.default_rng(0) if rng is None else rng

In [ ]:
if verbose:
        print("[1/4] Computing smoothed pairwise distances ...")
    distances = coordinates_to_distances(coord_list, bodypart_indices, parameters)

    if verbose:
        print("[2/4] Fitting online PCA ...")
    pca = fit_online_pca(distances, parameters, rng=rng)

    if verbose:
        print("[3/4] Building training set (wavelets + sub-sample t-SNE) ...")
    data_all = build_training_set(distances, pca, parameters, rng=rng, verbose=verbose)

    if verbose:
        print(f"[4/4] Final t-SNE of {data_all.shape[0]} training points ...")
    embedding = embed_training_set(data_all, parameters)